# Train/Test Split

## 1. Konfigurasi

In [1]:
from pathlib import Path

import pandas as pd


def find_base_dir(start=None) -> Path:
    """Cari root repo — folder pertama ke atas yang berisi `dataset/csv/`."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset" / "csv").is_dir():
            return candidate
    raise RuntimeError(f"Root repo tidak ditemukan dari {start}")


BASE_DIR = find_base_dir()

MODEL_READY_DIR = str(BASE_DIR / "dataset/model_ready")
FEATURED_FILE = Path(MODEL_READY_DIR) / "featured.parquet"

# Desember 2025 adalah test set terkunci — tidak ada fold walk-forward yang boleh menyentuhnya.
TEST_START = pd.Timestamp("2025-12-01")

print(f"BASE_DIR      = {BASE_DIR}")
print(f"FEATURED_FILE = {FEATURED_FILE}  ({'ada' if FEATURED_FILE.exists() else 'HILANG'})")
print(f"TEST_START    = {TEST_START.date()}")

BASE_DIR      = /Users/ramapdp/Project/Personal/forecast-scm
FEATURED_FILE = /Users/ramapdp/Project/Personal/forecast-scm/dataset/model_ready/featured.parquet  (ada)
TEST_START    = 2025-12-01


## 2. Load featured dataset

In [2]:
featured = pd.read_parquet(FEATURED_FILE)

print(featured.shape)
featured.head()

(1502522, 68)


,Kode Barang,Nama Cabang,Tanggal,Kuantitas,Kategori Barang,Nama Barang,Satuan,segment_id,day_of_week,day_of_month,...,roll_mean_7,roll_std_7,roll_mean_14,roll_std_14,roll_mean_28,roll_std_28,branch_avg_daily_qty,branch_demand_cv,branch_volume_tier,branch_age_days
0,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-01,235.0,Barang Jadi (FG),Ayam Kebuli (0.9),Potong,1,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,1670.017857,0.438671,flagship,0
1,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-02,147.0,Barang Jadi (FG),Ayam Kebuli (0.9),Potong,1,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,1670.017857,0.438671,flagship,1
2,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-03,85.0,Barang Jadi (FG),Ayam Kebuli (0.9),Potong,1,2,3,...,NaN,NaN,NaN,NaN,NaN,NaN,1670.017857,0.438671,flagship,2
3,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-04,106.0,Barang Jadi (FG),Ayam Kebuli (0.9),Potong,1,3,4,...,NaN,NaN,NaN,NaN,NaN,NaN,1670.017857,0.438671,flagship,3
4,FGS-00001,KY001 - Kebuli Yaman Kutabumi (Pusat),2024-01-05,155.0,Barang Jadi (FG),Ayam Kebuli (0.9),Potong,1,4,5,...,NaN,NaN,NaN,NaN,NaN,NaN,1670.017857,0.438671,flagship,4


## 3. Purging batas cutoff

In [3]:
# `target_lead_time_cumulative` menjumlahkan permintaan pada H+1 .. H+lead_time_days, jadi
# baris beberapa hari sebelum cutoff membawa label yang sebagian dibangun dari hari-hari
# setelahnya. Melatih model pada baris itu membocorkan periode test ke dalam label —
# volumenya kecil (lead_time_days maksimum 4 hari di sini) tapi cukup untuk menggugurkan
# klaim bahwa test set Desember benar-benar terkunci. Ini langkah purging standar untuk
# walk-forward validation pada target yang jendelanya menyeberangi titik split.
def lookahead_safe_mask(
    df: pd.DataFrame,
    boundary: pd.Timestamp,
    date_col: str = "Tanggal",
    lead_time_col: str = "lead_time_days",
) -> pd.Series:
    """True for rows whose whole target window stays strictly before `boundary`.

    A null lead time is safe: without one there is no lead-time target for the
    boundary to contaminate. Rows dated on or after the boundary come out
    False, which is harmless — callers combine this with their own date filter
    and never train on those rows anyway.
    """
    lead_time = pd.to_timedelta(df[lead_time_col].fillna(0), unit="D")
    return df[date_col] + lead_time < boundary

## 4. Split & export

In [4]:
def split_train_test(
    df: pd.DataFrame,
    cutoff: pd.Timestamp = TEST_START,
    date_col: str = "Tanggal",
    purge: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split on the cutoff, dropping train rows whose label crosses it.

    The last few days before the cutoff carry a lead-time target summed partly
    over test-period demand. Keeping them would train the model on labels built
    from the very month it is meant to be evaluated on. The test side is never
    purged — those rows are the evaluation, not the training data.
    """
    before_cutoff = df[date_col] < cutoff
    if purge and "lead_time_days" in df.columns:
        before_cutoff &= lookahead_safe_mask(df, cutoff, date_col=date_col)
    train = df[before_cutoff].reset_index(drop=True)
    test = df[df[date_col] >= cutoff].reset_index(drop=True)
    return train, test


def export_splits(train: pd.DataFrame, test: pd.DataFrame, output_dir: str = MODEL_READY_DIR) -> None:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    train.to_parquet(out / "train.parquet", index=False)
    test.to_parquet(out / "test.parquet", index=False)

## 5. Jalankan split & export

In [5]:
train, test = split_train_test(featured)
export_splits(train, test)

dibuang = len(featured) - len(train) - len(test)
print(f"train: {len(train):,} baris  |  test: {len(test):,} baris")
print(f"dibuang oleh purging: {dibuang:,} baris ({dibuang / len(featured):.4%} dari featured)")

train: 1,441,159 baris  |  test: 55,046 baris
dibuang oleh purging: 6,317 baris (0.4204% dari featured)


## 6. QA — integritas split

In [6]:
assert len(train) + len(test) + dibuang == len(featured), "train + test + purged ≠ featured"
assert train["Tanggal"].max() < TEST_START, "Ada tanggal train >= cutoff"
assert test["Tanggal"].min() >= TEST_START, "Ada tanggal test < cutoff"

# Purging: tidak boleh ada baris train yang jendela targetnya menyeberangi cutoff.
assert lookahead_safe_mask(train, TEST_START).all(), "Ada label train yang melewati cutoff"

# Baris yang dibuang harus persis baris pra-cutoff yang jendela targetnya menyeberang.
pra_cutoff = featured["Tanggal"] < TEST_START
assert dibuang == int((pra_cutoff & ~lookahead_safe_mask(featured, TEST_START)).sum()), (
    "jumlah baris terbuang tidak cocok dengan mask purging"
)

print(f"train: {train['Tanggal'].min().date()} … {train['Tanggal'].max().date()}")
print(f"test:  {test['Tanggal'].min().date()} … {test['Tanggal'].max().date()}")
print("Split integrity QA: OK")

train: 2024-01-01 … 2025-11-27
test:  2025-12-01 … 2025-12-31
Split integrity QA: OK


## 7. QA — NaN rate target di batas Desember 2025

In [7]:
# NaN rate mendekati boundary Des-2025 diharapkan: target_h{n}/target_lead_time_cumulative
# butuh n/lead_time_days hari ke depan yang datanya sudah tidak tersedia setelah 2025-12-31.
for h in range(1, 8):
    print(f"target_h{h} NaN rate (test): {test[f'target_h{h}'].isna().mean():.4f}")
print(f"target_lead_time_cumulative NaN rate (test): {test['target_lead_time_cumulative'].isna().mean():.4f}")

target_h1 NaN rate (test): 0.0349
target_h2 NaN rate (test): 0.0696
target_h3 NaN rate (test): 0.1043
target_h4 NaN rate (test): 0.1390
target_h5 NaN rate (test): 0.1736
target_h6 NaN rate (test): 0.2082
target_h7 NaN rate (test): 0.2426
target_lead_time_cumulative NaN rate (test): 0.0791


## 8. *(Opsional)* Cek sinkron dengan `utils/`

In [8]:
import inspect
import sys

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from utils.data_preprocessing import build_panel as _ref_panel
from utils.data_preprocessing import prepare_forecast_data as _ref_prep
from utils.modelling import purging as _ref_purging


def _kode(fn) -> str:
    """Kode sumber fungsi tanpa qualifier modul — di notebook semua fungsi satu namespace,
    jadi `purging.lookahead_safe_mask` (utils) dan `lookahead_safe_mask` (sini) bukan drift."""
    return inspect.getsource(fn).replace("purging.", "")


PASANGAN = [
    (lookahead_safe_mask, _ref_purging.lookahead_safe_mask),
    (split_train_test, _ref_prep.split_train_test),
    (export_splits, _ref_prep.export_splits),
]

beda = [nb.__name__ for nb, ref in PASANGAN if _kode(nb) != _kode(ref)]

# Konstanta ikut disalin dari utils, jadi ikut dibandingkan.
if TEST_START != _ref_panel.TEST_START:
    beda.append("TEST_START")
if MODEL_READY_DIR != _ref_prep.MODEL_READY_DIR:
    beda.append("MODEL_READY_DIR")

if beda:
    print("BERBEDA dari utils/ — salin ulang atau samakan: " + ", ".join(beda))
else:
    print(f"Sinkron: {len(PASANGAN)} fungsi + 2 konstanta identik dengan utils/")

Sinkron: 3 fungsi + 2 konstanta identik dengan utils/
